# 3. Glissants et xg_proxy

**Question** : comment les moyennes glissantes (buts, xg_proxy) sont-elles liées aux buts de l'équipe selon la demi-vie, et l'xg_proxy suit-il l'xG d'API-FOOTBALL ?

> Notebook **descriptif** (partie 3, ADR-0034). Il ne sert ni à choisir une variable ni à régler un paramètre :
> la sélection appartient au protocole de la partie 4 (validation glissante, ablation). Corrélation n'est pas apport.
> Données : période de développement seulement (matchs antérieurs au 1er juillet 2025, scellé de l'ADR-0012),
> lues par la porte `foot_predictor.features.sources`. Sorties effacées au commit (`nbstripout`) : réexécuter pour les voir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Seule porte vers les données (ADR-0028) : jeu versionné et, au besoin, matchs de la période de développement.
from foot_predictor.features.sources import load_dataset, load_matches

pd.set_option("display.width", 140)
NAMES = {39: "PL", 40: "Champ.", 140: "Liga", 141: "Segunda", 78: "BL", 79: "2. BL",
         135: "Serie A", 136: "Serie B", 61: "L1", 62: "L2"}
TOP5 = [39, 140, 78, 135, 61]
frame, manifest = load_dataset()
print("Jeu de données :", manifest["version"], "-", len(frame), "lignes, jusqu'au", frame["match_day"].max())
home = frame[frame["is_home"]].copy()  # une ligne par match (celle de l'équipe à domicile)
home["total"] = home["goals_for"] + home["goals_against"]

## Corrélation avec les buts de l'équipe, par demi-vie (top 5, 2015-16 à 2024-25)

In [ ]:
sub = frame[frame["eval_population"] & (frame["season_year"] >= 2015)]
rows = []
for h in (60, 120, 240):
    rows.append({"demi-vie": h,
                 "buts pour (équipe)": sub["goals_for"].corr(sub[f"goals_for_ewm_h{h}"]),
                 "buts contre (adversaire)": sub["goals_for"].corr(sub[f"opp_goals_against_ewm_h{h}"]),
                 "xg_proxy pour (équipe)": sub["goals_for"].corr(sub[f"xgp_for_ewm_h{h}"]),
                 "xg_proxy contre (adversaire)": sub["goals_for"].corr(sub[f"opp_xgp_against_ewm_h{h}"])})
display(pd.DataFrame(rows).set_index("demi-vie").round(3))
print("Pour comparaison, écart d'Elo :", round(sub["goals_for"].corr(sub["elo_diff"]), 3))

## xg_proxy contre xG d'API-FOOTBALL (top 5, 2022-23 à 2024-25)

In [ ]:
a = manifest["parameters"]["xg_proxy"]["on_target"]; b = manifest["parameters"]["xg_proxy"]["off_target"]
m = load_matches(competition_ids=None)
m = m[m["api_league_id"].isin(TOP5) & m["season_year"].between(2022, 2024) & (m["status"] == "played")]
pairs = []
for side in ("home", "away"):
    proxy = a * m[f"{side}_sot_fd"] + b * (m[f"{side}_shots_fd"] - m[f"{side}_sot_fd"])
    pairs.append(pd.DataFrame({"proxy": proxy.astype(float), "xg": m[f"{side}_xg_api"].astype(float),
                               "league": m["api_league_id"]}))
pairs = pd.concat(pairs).dropna()
print("lignes :", len(pairs), "- corrélation :", round(pairs["proxy"].corr(pairs["xg"]), 3))
display(pairs.groupby("league").apply(lambda g: pd.Series({"corr": g["proxy"].corr(g["xg"]),
        "proxy moyen": g["proxy"].mean(), "xG moyen": g["xg"].mean()})).rename(index=NAMES).round(3))
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pairs["xg"], pairs["proxy"], s=3, alpha=0.2); ax.plot([0, 5], [0, 5], color="grey", lw=0.8)
ax.set_xlabel("xG API-FOOTBALL"); ax.set_ylabel("xg_proxy"); ax.set_title("Par équipe et par match")
plt.show()

## Ancienneté et quantité d'information

Part des lignes dont le dernier match de championnat est ancien, et lien entre la somme des poids et la corrélation.

In [ ]:
d = frame["days_since_last_league_match"]
print("lignes sans match de championnat dans les 730 jours :", round(d.isna().mean() * 100, 2), "%")
print("lignes dont le dernier match date de plus de 120 jours :", round((d > 120).mean() * 100, 2), "%")
sub = sub.assign(poids=pd.cut(sub["goals_weight_h120"], [0, 2, 4, 6, 8, 20], include_lowest=True))
display(sub.groupby("poids", observed=True).apply(lambda g: pd.Series({
    "lignes": len(g), "corr buts pour": g["goals_for"].corr(g["goals_for_ewm_h120"])})).round(3))

## Conclusion

- **Corrélation avec les buts de l'équipe** (top 5, 2015-16 à 2024-25) : de 0,27 (h = 60 jours) à 0,29 (h = 240) pour les buts marqués lissés, de 0,27 à 0,28 pour l'xg_proxy lissé ; côté adversaire (buts ou xg_proxy concédés), de 0,16 à 0,17. Pour comparaison, l'écart d'Elo : 0,33. Les écarts entre demi-vies sont faibles ; le choix de la demi-vie reste à la validation interne de la partie 4, pas à ce tableau.
- **xg_proxy contre xG d'API-FOOTBALL** (2022-23 à 2024-25) : corrélation de 0,685 par équipe et par match (de 0,631 en Serie A à 0,725 en Bundesliga). L'xg_proxy moyen est un peu inférieur à l'xG, surtout en Premier League (1,42 contre 1,51).
- **Ancienneté** : 0,27 % des lignes n'ont aucun match de championnat dans les 730 jours (valeurs vides) et 0,07 % un dernier match vieux de plus de 120 jours. Dans ces championnats, l'ancienneté maximale ne touche presque personne. La somme des poids (h = 120) dépasse 8 dans presque toutes les lignes : les moyennes reposent sur beaucoup de matchs.